# Creating "All grid points of field" from the Meteorological Predictor Fields as Input for RF-based ML-Models
Version 15 December 2022, Selina Kiefer

### Input: csv-files
continuous timeseries of meteorological predictors as 2d-fields in csv-format
### Output: csv-file
continuous timeseries of all grid points of the meteorological predictors as a long vector per date in csv-format

#### Define the paths' and files' names

In [1]:
# Set the needed path and file names.
PATH_defined_functions = './Defined_Functions/'

PATH_input_data = './Data_in_csv_Format/'
ifiles_input_data = ['era5_u10_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z100_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z250_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_z500_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_z850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_t850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_H850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_u300_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_msl_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv']

PATH_output_file = './Predictors_All_Grid_Points_of_Field/'
file_name_output_file = 'era5_u10_z100_z250_z500_z850_t850_H850_u300_msl_60W_60E_20N_80N_1950_2020_lead_time_14d.csv'

#### Import the necessary packages and functions
Nothing needs to be changed here.

In [2]:
# Import the necessary python packages.
import yaml
import pandas as pd
import numpy as np

In [3]:
# Import the necessary functions.
import sys
sys.path.insert(1,PATH_defined_functions)
from read_in_csv_data import *

#### List the predictors to be combined

In [4]:
# List the desired predictors and set how many of these should be taken from the first 
# dataframe. From all other dataframes, only 1 predictor is taken (if more are needed, list
# these input files multiple times in "ifiles_input_data"). It is necessary to take the time as
# a predictor since the data will be grouped by date later.
desired_predictors = ['time', 'latitude', 'longitude', 'month', 'u10', 'z100', 'z250', 'z500', 'z850', 't850', 'H850', 'u300', 'msl']
number_of_predictors_in_first_dataframe = 5
time_column_name = 'time'

#### Combine all predictors into one dataframe
Nothing needs to be changed here.

In [5]:
# A new dataframe is created and the desired predictors from the first data file are written
# into it.
df_combined_input_data = pd.DataFrame()
df_input_data = read_in_csv_data(PATH_input_data, ifiles_input_data[0])
for i in range(number_of_predictors_in_first_dataframe):
    df_combined_input_data[desired_predictors[i]] = df_input_data [desired_predictors[i]]

In [6]:
# From all other dataframes, the specified predictor is added to this new dataframe.
for k in range(len(ifiles_input_data)-1):
    df_input_data = read_in_csv_data(PATH_input_data, ifiles_input_data[k+1])
    df_combined_input_data[desired_predictors[i+k+1]] = df_input_data [desired_predictors[i+k+1]]

In [7]:
# Now the time is set as the index and the data is grouped by date.
df_combined_input_data[time_column_name] = pd.to_datetime(df_combined_input_data[time_column_name])
df_combined_input_data = df_combined_input_data.set_index(time_column_name)
ds_input_data_grouped = df_combined_input_data.groupby([df_combined_input_data.index.year, df_combined_input_data.index.month, df_combined_input_data.index.day], as_index=False)

In [8]:
# The so grouped data is converted into a pandas dataframe.
df_input_data_grouped = pd.DataFrame(ds_input_data_grouped)

In [9]:
# Since the data is stacked, only the relevant column containing all the predictors is taken.
df_input_data_grouped = df_input_data_grouped[1]

#### Reshape the data so it can be used as input for RF-based ML-models directly
Nothing needs to be changed here.

In [10]:
# In a next step the data is reshaped so that the data can be used directly with the machine 
# learning model later. Therefore, it needs to have in 1 dimension the same length as the 
# ground truth data. Here, this is the time. So for every date, all predictors are put into a 
# separate row and then appended to a list. The predictors are thereby sorted in the same way
# in each row.
reshaped_data = []
for l in range(len(df_input_data_grouped)):
    single_day = np.array(df_input_data_grouped[l])
    single_day = single_day.reshape(1,-1)
    reshaped_data.append(single_day)

In [11]:
# The so created list containing all the predictors is converted into a pandas dataframe 
# again.
df_reshaped_data = pd.DataFrame(np.squeeze(reshaped_data))

#### Add the time information again to the reshaped data
Nothing needs to be changed here.

In [12]:
# Since the time got lost by using .groupby(), a separate new dataframe is created containing
# only the time. To this dataframe, three new columns are added containing the year, month and 
# day.
df_combined_input_data = df_combined_input_data.reset_index()
df_time = pd.DataFrame()
df_time[time_column_name] = pd.to_datetime(df_combined_input_data[time_column_name])
df_time = df_time.set_index(time_column_name)
df_time['year'] = df_time.index.year
df_time['month'] = df_time.index.month
df_time['day'] = df_time.index.day
df_time = df_time.reset_index()

In [13]:
# This new dataframe is then grouped by date and 'averaged' resulting in a daily time-
# series but separated into year, month and day.
df_time = df_time.set_index(time_column_name)
ds_time_mean = df_time.groupby([df_time.index.year, df_time.index.month, df_time.index.day], as_index=False).mean()
df_time_mean = pd.DataFrame(ds_time_mean)

In [14]:
# The separated timeseries is now combined into a single daily timeseries.
daily_timeseries = (df_time_mean['year'].astype(str)+'-'+df_time_mean['month'].astype(str)+'-'+df_time_mean['day'].astype(str))

In [15]:
# In the next step, the time is added to the dataframe containing the predictors.
df_reshaped_data.insert(0, time_column_name, daily_timeseries)

#### Doublecheck the representation of the data

In [16]:
# Check if everything is reshaped and sorted correctly.
df_reshaped_data.head()

,time,0,1,2,3,4,5,6,7,8,...,38870,38871,38872,38873,38874,38875,38876,38877,38878,38879
0,1950-1-1,79.5,-60.0,1.0,8.750118,151657.303091,94083.768989,49628.829258,13356.456170,245.789448,...,1.0,25.150882,161361.408247,106172.899771,57029.960865,14758.520435,286.229888,52.337978,32.181980,101415.263142
1,1950-1-2,79.5,-60.0,1.0,11.137983,152708.168132,95714.591387,51251.104163,13126.925910,251.971954,...,1.0,26.197250,161092.356984,105811.167159,56848.646449,14649.595123,285.361767,40.771117,38.384167,101344.415592
2,1950-1-3,79.5,-60.0,1.0,10.842854,152905.671136,95337.730553,51369.629647,13293.487692,255.562947,...,1.0,22.659529,160787.158946,105439.012883,56833.176712,14641.216253,282.814518,71.939402,26.277139,101361.143486
3,1950-1-4,79.5,-60.0,1.0,5.043752,152529.297487,94525.849273,50641.642030,12782.122710,252.781534,...,1.0,22.605870,160701.450095,105367.069780,56868.666109,14865.668410,282.486117,75.450145,31.316978,101646.501674
4,1950-1-5,79.5,-60.0,1.0,7.922222,151280.929441,93355.597208,49193.856657,12294.370908,249.316189,...,1.0,18.389735,161021.554020,105724.432016,57053.165470,15010.902158,284.505069,20.366776,26.307173,101848.548391


In [17]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_reshaped_data.tail()

,time,0,1,2,3,4,5,6,7,8,...,38870,38871,38872,38873,38874,38875,38876,38877,38878,38879
12850,2020-12-13,79.5,-60.0,12.0,-0.137680,150748.683751,96397.377980,51856.909900,13209.530050,260.932063,...,12.0,-11.602207,162119.814938,106826.330830,57188.818055,14902.474638,283.552567,48.656907,31.716550,101617.384757
12851,2020-12-14,79.5,-60.0,12.0,-2.244191,150768.113279,96108.356903,51982.393019,13490.493067,261.331091,...,12.0,-16.019497,161842.944166,106624.931528,57167.454441,14948.964922,284.697481,20.162227,33.658561,101700.053045
12852,2020-12-15,79.5,-60.0,12.0,-3.865566,150505.067363,95406.074919,51683.752191,14332.647094,253.106606,...,12.0,-14.708779,161817.909966,106772.711536,57331.392064,15093.213430,284.540972,26.449655,30.430163,101869.597287
12853,2020-12-16,79.5,-60.0,12.0,-5.023084,150314.882177,95417.191115,51465.618454,13781.930179,254.199350,...,12.0,-17.172759,161793.623056,106895.316631,57499.827290,15182.702631,286.816699,11.209929,26.959401,101949.047946
12854,2020-12-17,79.5,-60.0,12.0,-11.261761,149623.265713,94881.652060,50803.571317,13119.673336,252.830248,...,12.0,-18.960102,161798.106794,106881.911807,57587.980306,15186.561508,288.276041,7.415264,22.009605,101943.107710


#### Save the data in csv format
Nothing needs to be changed here.

In [18]:
# Save the pandas dataframe in csv-format.
df_reshaped_data.to_csv(PATH_output_file+file_name_output_file)

In [19]:
# End of Program